# Imputacion valores NAN y OUTLIERS segun rango temporal con metodo por Transecto y metodo General

In [25]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2 (adaptado a transectos) - Versión corregida para evitar NaNs residuales.
Corrección principal: se reemplazaron llamadas a fillna(method='ffill') por .ffill().bfill()
Entrada: Archivos *_outliers.csv en INPUT_DIR
Salida:
  - imputed_by_transect/: archivos Transecto_1.csv, Transecto_2.csv, ...
  - imputed_global/     : archivos por estación
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
INPUT_DIR = BASE_DIR  # donde están los *_outliers.csv

OUTPUT_BY_TRANSECT = os.path.join(BASE_DIR, "imputed_by_transect")
OUTPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_TRANSECT, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales (horas)
SHORT_GAP_MAX = 6
MEDIUM_GAP_MAX = 72
LONG_GAP_MIN = 72

# Método para imputación estacional ('mean' o 'median')
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye tanto O3_for_impute como O3 por seguridad)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute', 'O3',
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Para KNN
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filepath):
    """Carga un CSV con índice datetime."""
    df = pd.read_csv(filepath, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un dict {tuple(indice): longitud} para cada grupo consecutivo de NaN en 'series'.
    """
    mask = series.isna()
    if mask.empty:
        return {}
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)
    return gap_groups

def compute_hourly_stats(series, stat='mean'):
    """
    Calcula el estadístico horario (media o mediana) y asegura que existan valores para todas las horas (0-23).
    Si no hay datos para una hora, se rellena con el valor más cercano (ffill/bfill).
    Si la serie está vacía, retorna Serie de NaN.
    """
    if series is None or series.empty:
        return pd.Series(np.nan, index=range(24))
    if stat == 'mean':
        hourly = series.groupby(series.index.hour).mean()
    else:
        hourly = series.groupby(series.index.hour).median()
    hourly = hourly.reindex(range(24))
    hourly = hourly.ffill().bfill()
    return hourly

def impute_dataframe(df, hourly_stats_dict):
    """
    Imputa un DataFrame de una estación:
      - Gaps cortos (<= SHORT_GAP_MAX): interpolación lineal.
      - Gaps medios (< MEDIUM_GAP_MAX): imputación por estadístico horario.
      - Gaps largos (>= MEDIUM_GAP_MAX): se dejan para KNN.
    """
    df_imp = df.copy()
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        df_imp[col] = pd.to_numeric(df_imp[col], errors='coerce')
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        gaps = get_gap_lengths(serie)
        if not gaps:
            continue

        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)
            if L <= SHORT_GAP_MAX:
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                # Asegurar que las posiciones existan antes de asignar
                common_idx = serie_interp.index.intersection(idx)
                if not common_idx.empty:
                    df_imp.loc[common_idx, col] = serie_interp.loc[common_idx]
            elif L < MEDIUM_GAP_MAX:
                if col in hourly_stats_dict:
                    # map horas a valores horarios
                    hours = pd.Index(idx).hour
                    hourly_series = hourly_stats_dict[col]
                    # si hourly_series es una serie con índice 0-23, indexarlo
                    vals = [hourly_series.get(h, np.nan) if hasattr(hourly_series, "get") else hourly_series.loc[h] for h in hours]
                    df_imp.loc[idx, col] = vals
                # si no hay estadístico, se deja NaN para KNN
            else:
                # gap largo: dejamos NaN para KNN
                pass
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas num_cols (solo las que existan en df),
    usando extra_cols como predictores (deben ser numéricos).
    Retorna df con las columnas imputadas.
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if not cols_to_impute:
        return df

    if extra_cols:
        extra_cols = [c for c in extra_cols if c in df.columns]
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    sub = df[all_cols].copy()
    for c in sub.columns:
        sub[c] = pd.to_numeric(sub[c], errors='coerce')

    if extra_cols:
        for col in extra_cols:
            if sub[col].isna().any():
                sub[col].fillna(-999, inplace=True)

    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    try:
        imputed_array = imputer.fit_transform(sub)
    except Exception as e:
        print("  ERROR en KNNImputer:", e)
        print("  Intentando imputación de fallback por media de columna.")
        for c in cols_to_impute:
            mean_val = sub[c].mean(skipna=True)
            sub[c].fillna(mean_val if not pd.isna(mean_val) else 0.0, inplace=True)
        imputed_array = sub.values

    df_imp = df.copy()
    df_imp.loc[:, cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

def fill_remaining_nans_with_mean(df, cols):
    """
    Rellena los NaN que aún puedan quedar en las columnas especificadas con la media de cada columna.
    Si una columna es completamente NaN, se rellena con 0 (último recurso).
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            mean_val = pd.to_numeric(df[col], errors='coerce').mean(skipna=True)
            if pd.isna(mean_val):
                mean_val = 0.0
            df[col] = df[col].fillna(mean_val)
    return df

def ensure_no_nans(df, cols):
    """
    Verifica que no queden NaN en cols; si quedan, rellena con 0 e imprime advertencia.
    """
    for col in cols:
        if col in df.columns and df[col].isna().any():
            n_nans = int(df[col].isna().sum())
            print(f"  ADVERTENCIA: {n_nans} NaN residuales en '{col}' rellenados con 0.")
            df[col] = df[col].fillna(0)
    return df

# ============================================================================
# PROCESAMIENTO POR TRANSECTO (MEJORADO)
# ============================================================================

def process_by_transect():
    print("\n=== Imputación por transecto (agrupando estaciones) ===")

    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    transect_dict = {}

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' not in df.columns:
            print(f"  Advertencia: {filepath.name} no tiene columna 'Transecto'. Se omite.")
            continue
        transect_names = df['Transecto'].dropna().unique()
        if len(transect_names) == 0:
            print(f"  Advertencia: {filepath.name} no tiene valores en Transecto. Se omite.")
            continue
        transect = transect_names[0]
        df['Transecto'] = df['Transecto'].fillna(transect)

        transect_clean = str(transect).replace(" ", "_")

        if transect_clean not in transect_dict:
            transect_dict[transect_clean] = []
        transect_dict[transect_clean].append((station_name, df))

    for transect_clean, station_list in transect_dict.items():
        print(f"\nProcesando transecto: {transect_clean}")

        df_all = pd.concat([df for _, df in station_list], axis=0, sort=False)
        df_all = df_all.sort_index()

        hourly_stats = {}
        for col in NUM_COLS:
            if col not in df_all.columns:
                continue
            s = pd.to_numeric(df_all[col], errors='coerce').dropna()
            hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

        df_imputed_list = []
        for station_name, df_station in station_list:
            df_station['Estacion'] = df_station['Estacion'].fillna(station_name)
            if 'Transecto' in df_station.columns:
                df_station['Transecto'] = df_station['Transecto'].fillna(transect.replace("_", " "))
            for col in NUM_COLS:
                if col in df_station.columns:
                    df_station[col] = pd.to_numeric(df_station[col], errors='coerce')

            df_station_imp = impute_dataframe(df_station, hourly_stats)
            df_imputed_list.append(df_station_imp)

        df_concat = pd.concat(df_imputed_list, axis=0, sort=False)
        df_concat = df_concat.sort_index()

        # Reemplazamos fillna(method=...) por ffill().bfill()
        if 'Estacion' in df_concat.columns:
            df_concat['Estacion'] = df_concat['Estacion'].ffill().bfill().fillna('unknown_station')
        if 'Transecto' in df_concat.columns:
            df_concat['Transecto'] = df_concat['Transecto'].ffill().bfill().fillna(transect_clean.replace("_", " "))

        df_concat['station_code'] = pd.factorize(df_concat['Estacion'])[0]
        extra_cols = ['station_code']

        df_concat = apply_knn_imputation(df_concat, NUM_COLS, extra_cols=extra_cols)
        df_concat = fill_remaining_nans_with_mean(df_concat, NUM_COLS)
        df_concat = ensure_no_nans(df_concat, NUM_COLS)

        if 'station_code' in df_concat.columns:
            df_concat.drop(columns=['station_code'], inplace=True)

        if 'O3_for_impute' in df_concat.columns:
            df_concat['O3'] = df_concat['O3_for_impute']
            df_concat.drop(columns=['O3_for_impute'], inplace=True)

        out_path = os.path.join(OUTPUT_BY_TRANSECT, f"{transect_clean}.csv")
        df_concat.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path} (shape: {df_concat.shape})")

# ============================================================================
# PROCESAMIENTO GLOBAL (MEJORADO)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)

        if 'O3' in df.columns and 'O3_for_impute' not in df.columns:
            df['O3_for_impute'] = pd.to_numeric(df['O3'], errors='coerce')

        if 'Estacion' not in df.columns:
            df['Estacion'] = station_name
        else:
            df['Estacion'] = df['Estacion'].fillna(station_name)

        if 'Transecto' in df.columns:
            tran_names = df['Transecto'].dropna().unique()
            if len(tran_names) > 0:
                df['Transecto'] = df['Transecto'].fillna(tran_names[0])

        df['station_id'] = station_name
        dfs.append(df)

    df_all = pd.concat(dfs, axis=0, sort=False)
    df_all = df_all.sort_index()

    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = pd.to_numeric(df_all[col], errors='coerce').dropna()
        global_hourly_stats[col] = compute_hourly_stats(s, SEASONAL_STAT)

    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        for col in NUM_COLS:
            if col in df_station.columns:
                df_station[col] = pd.to_numeric(df_station[col], errors='coerce')
        df_station = impute_dataframe(df_station, global_hourly_stats)
        df_all_imp.loc[mask] = df_station

    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    df_all_imp = fill_remaining_nans_with_mean(df_all_imp, NUM_COLS)
    df_all_imp = ensure_no_nans(df_all_imp, NUM_COLS)

    if 'station_code' in df_all_imp.columns:
        df_all_imp.drop(columns=['station_code'], inplace=True)

    if 'O3_for_impute' in df_all_imp.columns:
        df_all_imp['O3'] = df_all_imp['O3_for_impute']
        df_all_imp.drop(columns=['O3_for_impute'], inplace=True)

    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        if 'station_id' in df_station.columns:
            df_station.drop(columns=['station_id'], inplace=True)
        if 'Estacion' not in df_station.columns or df_station['Estacion'].isna().any():
            df_station['Estacion'] = df_station['Estacion'].ffill().bfill().fillna(station)
        if 'Transecto' in df_station.columns:
            df_station['Transecto'] = df_station['Transecto'].ffill().bfill().fillna('unknown_transect')

        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado: {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos (por transecto y global)...")
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_TRANSECT}")
    print(f"  - {OUTPUT_GLOBAL}")

OSError: [Errno 30] Read-only file system: '/usu'